In [2]:
import os
import torch
# from model import DelphiConfig, Delphi
from tqdm import tqdm
import pandas as pd
import numpy as np
import textwrap
import matplotlib.pyplot as plt
%config InlineBackend.figure_format='retina'

In [3]:
device = 'cpu'

ckpt_path = 'out/0311_0133_out_v6/ckpt.pt'
checkpoint = torch.load(ckpt_path, map_location=device)
model_args = dict(checkpoint["model_args"])

In [7]:
from model_v6 import CompositeDelphi, CompositeDelphiConfig

conf = CompositeDelphiConfig(**model_args)
model = CompositeDelphi(conf)
model

CompositeDelphi(
  (composite_emb): CompositeEmbedding(
    (data_emb): Embedding(1290, 256)
    (shift_emb): Embedding(5, 256)
    (total_emb): Embedding(552, 256)
    (shift_value_proj): Sequential(
      (0): Linear(in_features=1, out_features=256, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=256, out_features=256, bias=True)
    )
    (proj): Linear(in_features=768, out_features=256, bias=False)
  )
  (age_encoding): AgeEncoding(
    (linear): Linear(in_features=256, out_features=256, bias=False)
  )
  (token_drop): Dropout(p=0.0, inplace=False)
  (drop): Dropout(p=0.2, inplace=False)
  (h): ModuleList(
    (0-7): 8 x TransformerBlock(
      (ln_1): RMSNorm()
      (attn): GroupedQueryAttention(
        (q_proj): Linear(in_features=256, out_features=256, bias=False)
        (k_proj): Linear(in_features=256, out_features=128, bias=False)
        (v_proj): Linear(in_features=256, out_features=128, bias=False)
        (o_proj): Linear(in_features=256, o

In [9]:
state_dict = checkpoint['model']
model.load_state_dict(state_dict)

<All keys matched successfully>

In [11]:
model.eval()
model = model.to(device)

checkpoint['model_args']

{'n_layer': 8,
 'n_head': 8,
 'n_kv_head': 4,
 'n_embd': 256,
 'block_size': 512,
 'bias': False,
 'dropout': 0.2,
 'token_dropout': 0.0,
 't_min': 0.1,
 'mask_ties': True,
 'ignore_tokens': [0],
 'use_moe': True,
 'num_experts': 8,
 'experts_per_token': 2,
 'sliding_window': 128,
 'rope_theta': 10000.0,
 'use_drug_conditioning': True,
 'drug_token_min': 1278,
 'drug_token_max': 1288,
 'mdn_n_components': 8,
 'shift_min_value': 0.0,
 'shift_max_value': 500,
 'shift_continuous': True,
 'shift_log': True,
 'shift_input_scale': 20.794416427612305,
 'shift_exclude_na_token': True,
 'shift_mdn_nll_weight': 0.05,
 'total_min_value': 0.0,
 'total_max_value': 550.0,
 'total_log_transform': False,
 'data_vocab_size': 1290,
 'shift_vocab_size': 5,
 'total_vocab_size': 552,
 'shift_loss_type': 'dice_focal',
 'shift_dice_weight': 0.5,
 'shift_ignore_index': -1,
 'shift_maintain_idx': 2,
 'shift_change_weight_max': 10.0,
 'shift_focal_gamma': 2.0,
 'shift_class_weights': [],
 'apply_token_shift': F

In [15]:
import numpy as np
from collections import Counter

composite_dtype = np.dtype([
    ('ID', np.uint32),
    ('AGE', np.uint32),
    ('DATA', np.uint32),
    ('SHIFT', np.float32),
    ('TOTAL', np.uint32)
])

train_data = np.fromfile('../data/dose/JMDC_exval2.bin', dtype=composite_dtype)
train_data

array([(       1,    22122,        5, 0.0000e+00,   0),
       (       0,        1,    22122, 1.2612e-44,   0),
       (       0,        0,        1, 3.1000e-41,  10), ...,
       (30373340,    20760,      508, 0.0000e+00,   0),
       (       0, 30373340,    20760, 3.8536e-43,   0),
       (       0,        0, 30373340, 2.9175e-41, 392)],
      shape=(402355221,), dtype=[('ID', '<u4'), ('AGE', '<u4'), ('DATA', '<u4'), ('SHIFT', '<f4'), ('TOTAL', '<u4')])

In [16]:
example_health_trajectory = [
    ('Male', 0.0),
    ('No event', 5.0),
    ('No event', 10.0),
    ('No event', 15.0),
    ('No event', 20.0),
    ('No event', 25.0),
    ('No event', 30.0),
    ('No event', 35.0),
    ('No event', 40.0),

    ('J04 (acute laryngitis and tracheitis)', 41.3),
    ('J03 (acute tonsillitis)', 41.7),
    ('K28 (gastrojejunal ulcer)', 42.6),
    ('E12 (malnutrition-related diabetes mellitus)', 43.1),

    ('DPP-4 inhibitor', 43.6),
    ('Sulfonylurea', 43.7),
    ('GLP-1', 43.7),
    ('Alpha-glucosidase inhibitors', 43.7),

    ('K30 (dyspepsia)', 43.8),
    ('K06 (other disorders of gingiva and edentulous alveolar ridge)', 44.2),
    ('Thiazolidinedione', 44.4),
    ('E12 (malnutrition-related diabetes mellitus)', 44.9),
    ('No event', 45.5),
]

# 기존 notebook 방식 그대로
example_health_trajectory = [(event, age_year * 365.25) for event, age_year in example_health_trajectory]

In [22]:
labels = pd.read_csv('../data/labels.csv')

In [27]:
labels

,Padding,Unnamed: 1
0,No event,NaN
1,Female,NaN
2,Male,NaN
3,BMI_low,NaN
4,BMI_mid,NaN
...,...,...
1283,Alpha-glucosidase inhibitors,NaN
1284,GLP-1,NaN
1285,SGLT-2 inhibitor,NaN
1286,Other,NaN


In [29]:
max_new_tokens = 100

# event name -> token id
name_to_token_id = {name: idx for idx, name in labels['Padding'].items()}

# token id -> event name
token_id_to_name = {idx: name for idx, name in labels['Padding'].items()}

events = [name_to_token_id[event[0]] for event in example_health_trajectory]
events = torch.tensor(events, device=device).unsqueeze(0)

ages = [event[1] for event in example_health_trajectory]
ages = torch.tensor(ages, device=device).unsqueeze(0)

with torch.no_grad():
    y, b, _ = model.generate(events, ages, max_new_tokens, termination_tokens=[1290])
    events_data = zip(y.cpu().numpy().flatten(), b.cpu().numpy().flatten() / 365.)

    print('Input trajectory:')
    for i, (event_id, age_years) in enumerate(events_data):
        if i == len(example_health_trajectory):
            print('=====================')
            print('Generated trajectory:')

        event_name = token_id_to_name.get(event_id, f"<UNK:{event_id}>")
        print(f"{age_years:2.1f}: {event_name}")

TypeError: CompositeDelphi.generate() missing 1 required positional argument: 'age'